# ML pipeline

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import pickle


df = pd.read_csv('Titanic-Dataset.csv')


df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)


X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'FamilySize', 'IsAlone']]
y = df['Survived']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (712, 9)
Testing shape: (179, 9)


In [2]:
df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize,IsAlone
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2,0


# Making and Saving the pipeline

In [3]:
from sklearn.pipeline import make_pipeline

num_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']
cat_features = ['Sex', 'Embarked']

num_transformer = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)

cat_transformer = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore')
)

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

titanic_pipeline = make_pipeline(
    preprocessor,
    LogisticRegression(random_state=42)
)

titanic_pipeline.fit(X_train, y_train)

print("Pipeline successfully fitted!")

Pipeline successfully fitted!


# Evaluation Metrics of the Pipeline

In [4]:
import pickle
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Make predictions using the fitted pipeline directly on raw test features
y_pred = titanic_pipeline.predict(X_test)

print("=== PIPELINE EVALUATION RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Save the complete pipeline (Preprocessing + Model) using Pickle
model_filename = 'titanic_pipeline.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(titanic_pipeline, file)

print(f"\nPipeline successfully saved to {model_filename}!")




=== PIPELINE EVALUATION RESULTS ===
Accuracy: 0.8044692737430168

Confusion Matrix:
 [[97 13]
 [22 47]]

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.88      0.85       110
           1       0.78      0.68      0.73        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179


Pipeline successfully saved to titanic_pipeline.pkl!


 # Giving real data to the model for testing

In [6]:
import pickle
import pandas as pd



with open(model_filename, 'rb') as file:
    loaded_pipeline = pickle.load(file)


sample_data = pd.DataFrame([{
    'Pclass': 3,
    'Sex': 'male',
    'Age': 22.0,
    'SibSp': 1,
    'Parch': 0,
    'Fare': 7.25,
    'Embarked': 'S',
    'FamilySize': 2,
    'IsAlone': 0
}])


sample_prediction = loaded_pipeline.predict(sample_data)
sample_probability = loaded_pipeline.predict_proba(sample_data)

print("\n=== LOADED PIPELINE PREDICTION ===")
print("Predicted Class (0 = Died, 1 = Survived):", sample_prediction[0])
print(f"Prediction Probability: {sample_probability[0][sample_prediction[0]]:.2%}")


=== LOADED PIPELINE PREDICTION ===
Predicted Class (0 = Died, 1 = Survived): 0
Prediction Probability: 87.18%


# Making the pipeline without the engineered features 

In [11]:

X_baseline = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]
X_tr_base, X_te_base, y_tr_base, y_te_base = train_test_split(
    X_baseline, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor_base = ColumnTransformer(transformers=[
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), ['Sex', 'Embarked'])
])

baseline_pipeline = make_pipeline(preprocessor_base, LogisticRegression(random_state=42))
baseline_pipeline.fit(X_tr_base, y_tr_base)

acc_baseline = accuracy_score(y_te_base, baseline_pipeline.predict(X_te_base))
acc_pipeline = 0.8045

print("=== FEATURE ENGINEERING IMPACT ===")
print(f"Baseline Accuracy (Without Engineered Features): {acc_baseline:.4f}")
print(f"Pipeline Accuracy (With FamilySize & IsAlone):   {acc_pipeline:.4f}")
print(f"Accuracy Delta:                                  {acc_pipeline - acc_baseline:+.4f}")

=== FEATURE ENGINEERING IMPACT ===
Baseline Accuracy (Without Engineered Features): 0.8045
Pipeline Accuracy (With FamilySize & IsAlone):   0.8045
Accuracy Delta:                                  +0.0000


### Feature Engineering Impact Analysis
- **Baseline Model (without FamilySize & IsAlone):** 80.45% Accuracy
- **Pipeline Model (with FamilySize & IsAlone):** 80.45% Accuracy
- **Accuracy Delta:** 0.00%

**Technical Observation:**
The engineered features (`FamilySize` and `IsAlone`) provided zero net accuracy gain for linear Logistic Regression.Because `FamilySize` is a direct linear combination of `SibSp + Parch + 1`, 
a linear model already assigns individual weights to `SibSp` and `Parch` that approximate the same underlying decision boundary. However,
keeping these features inside the automated pipeline remains valuable for non-linear tree-based models (like Decision Trees or Random Forests) where explicit thresholds on family size often improve split efficiency.